# MLP 레이어 모듈

이 노트북은 `src/nn/layers.py`가 제공하는 레이어 모듈을 직접 실행하고 forward/backward 동작을 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**목표**
- `Module`, `Linear`, `Sigmoid`, `ReLU`, `Sequential`의 인터페이스를 직접 확인한다.
- `Linear`의 He 초기화, forward/backward shape, gradient 계산을 검증한다.
- `Sequential`로 레이어를 체인으로 연결하고 params/grads 수집 방식을 확인한다.
- forward/backward를 한 스텝 수동으로 실행하여 파라미터 업데이트 흐름을 추적한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.nn.layers import Module, Linear, Sigmoid, ReLU, Sequential
from src.nn.losses import cross_entropy, cross_entropy_grad

## 1. 개요

`src/nn/layers.py`는 MLP를 구성하는 레이어 모듈을 제공한다.
모든 레이어는 `Module` 기반이며 `forward`와 `backward` 메서드를 가진다.
파라미터가 있는 레이어는 `params`와 `grads` 리스트를 통해 optimizer와 연동한다.

| 이름 | 역할 | params/grads |
|---|---|---|
| `Module` | 기반 클래스. forward/backward/train/eval 인터페이스 정의 | 빈 리스트 |
| `Linear` | 완전 연결 레이어 $y = xW + b$ | `[w, b]` / `[grad_w, grad_b]` |
| `Sigmoid` | sigmoid 활성화 레이어 | 빈 리스트 |
| `ReLU` | ReLU 활성화 레이어 | 빈 리스트 |
| `Sequential` | 레이어 체인 컨테이너 | 하위 레이어 params/grads 자동 집계 |

## 2. Linear

완전 연결 레이어이다. 입력 $x$에 가중치 $W$를 곱하고 편향 $b$를 더하는 아핀 변환을 수행한다.

**Forward**: $y = xW + b$, $x \in \mathbb{R}^{N \times D_{in}}$, $W \in \mathbb{R}^{D_{in} \times D_{out}}$

**Backward** (chain rule):

$$
\frac{\partial L}{\partial W} = x^T \cdot \text{dout}, \quad
\frac{\partial L}{\partial b} = \sum_i \text{dout}_i, \quad
\frac{\partial L}{\partial x} = \text{dout} \cdot W^T
$$

**He 초기화**: $W \sim \mathcal{N}(0,\ \sqrt{2/D_{in}})$

In [ ]:
layer = Linear(784, 256, seed=42)

print("=== Linear(784, 256) ===")
print(f"w shape:      {layer.w.shape}")
print(f"b shape:      {layer.b.shape}")
print(f"grad_w shape: {layer.grad_w.shape}")
print(f"grad_b shape: {layer.grad_b.shape}")
print(f"params count: {len(layer.params)}  (w, b)")
print(f"grads  count: {len(layer.grads)}   (grad_w, grad_b)")
print()
# He 초기화: 표준편차 ≈ sqrt(2/784)
expected_std = np.sqrt(2.0 / 784)
print(f"w std: {layer.w.std():.4f}  (expected ≈ {expected_std:.4f})")
print(f"b init: {layer.b[:5]}  (must be zeros)")

In [ ]:
N = 32
x = np.random.randn(N, 784).astype(np.float32)

out = layer.forward(x)
print(f"input  shape: {x.shape}")
print(f"output shape: {out.shape}  <- (N, 256) 기대")

In [ ]:
dout = np.random.randn(N, 256).astype(np.float32)
dx = layer.backward(dout)

print(f"dout shape:   {dout.shape}")
print(f"dx shape:     {dx.shape}      <- (N, 784) 기대")
print(f"grad_w shape: {layer.grad_w.shape}  <- (784, 256) 기대")
print(f"grad_b shape: {layer.grad_b.shape}  <- (256,) 기대")
print()
# chain rule 검증
expected_grad_w = x.T @ dout
expected_grad_b = dout.sum(axis=0)
expected_dx = dout @ layer.w.T
print(f"grad_w matches x.T @ dout:     {np.allclose(layer.grad_w, expected_grad_w)}")
print(f"grad_b matches dout.sum(axis0): {np.allclose(layer.grad_b, expected_grad_b)}")
print(f"dx matches dout @ w.T:          {np.allclose(dx, expected_dx)}")

In [ ]:
# in-place 대입 검증: grads 리스트의 참조가 backward 후에도 유지되어야 한다
grad_w_ref = layer.grads[0]  # params list의 참조
_ = layer.forward(x)
_ = layer.backward(dout)
print(f"grads[0] is layer.grad_w: {layer.grads[0] is layer.grad_w}")
print(f"grad_w_ref is grads[0]:   {grad_w_ref is layer.grads[0]}")
print("→ in-place 대입으로 배열 참조가 바뀌지 않음")

## 3. Sigmoid 레이어

활성화 레이어이다. forward에서 출력을 `self._out`에 저장하고, backward에서 재사용한다.

**Backward**:

$$
\frac{\partial L}{\partial x} = \text{dout} \cdot \sigma(x) \cdot (1 - \sigma(x))
$$

In [ ]:
sig = Sigmoid()

x = np.array([[-2.0, 0.0, 2.0]], dtype=np.float32)
out = sig.forward(x)
print(f"input:   {x}")
print(f"output:  {out.round(4)}  <- must be in (0, 1)")

dout = np.ones_like(x)
dx = sig.backward(dout)
# sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))
expected_dx = out * (1.0 - out)
print(f"\ndx:          {dx.round(4)}")
print(f"expected_dx: {expected_dx.round(4)}")
print(f"match: {np.allclose(dx, expected_dx)}")
print(f"sigmoid'(0) = {float(dx[0, 1]):.4f}  <- 0.25 기대")

## 4. ReLU 레이어

활성화 레이어이다. forward에서 `self._mask = x > 0`으로 boolean 마스크를 저장하고, backward에서 그대로 곱한다.

**Backward**:

$$
\frac{\partial L}{\partial x} = \text{dout} \cdot \mathbf{1}[x > 0]
$$

In [ ]:
r = ReLU()

x = np.array([[-2.0, -1.0, 0.0, 1.0, 2.0]], dtype=np.float32)
out = r.forward(x)
print(f"input:   {x}")
print(f"output:  {out}  <- 음수=0, 양수=그대로")

dout = np.ones_like(x)
dx = r.backward(dout)
print(f"\ndx:   {dx}")
print(f"mask: {r._mask}  <- x > 0 위치만 gradient 통과")
print(f"음수 위치 gradient = 0:  {np.all(dx[x <= 0] == 0)}")
print(f"양수 위치 gradient = 1:  {np.all(dx[x > 0] == 1)}")

## 5. Sequential

여러 레이어를 순서대로 연결하는 컨테이너이다.
forward는 순방향, backward는 역방향으로 레이어를 순회한다.
`params`와 `grads`는 생성자에서 하위 레이어 리스트를 `extend`로 수집한다.

In [ ]:
model = Sequential(
    Linear(784, 256, seed=0),
    Sigmoid(),
    Linear(256, 128, seed=1),
    Sigmoid(),
    Linear(128, 10, seed=2),
)

print(f"레이어 수: {len(model.layers)}")
print(f"params 수: {len(model.params)}  <- Linear 3개 × (w, b) = 6")
print()

# shape 추적
print("=== 파라미터 shape 추적 ===")
for i, (p, g) in enumerate(zip(model.params, model.grads)):
    print(f"  params[{i}] shape: {p.shape}")

In [ ]:
N = 32
x = np.random.randn(N, 784).astype(np.float32)
logits = model.forward(x)

print(f"input  shape: {x.shape}")
print(f"output shape: {logits.shape}  <- (32, 10) 기대")

In [ ]:
# 간단한 one-hot target으로 loss/grad 계산
targets = np.eye(10, dtype=np.float32)[np.arange(N) % 10]

loss = cross_entropy(logits, targets)
grad = cross_entropy_grad(logits, targets)

model.backward(grad)

print(f"loss: {loss:.4f}")
print(f"\n=== backward 후 grad 확인 ===")
for i, g in enumerate(model.grads):
    print(f"  grads[{i}] norm: {np.linalg.norm(g):.4f}  shape: {g.shape}")

## 6. 수동 파라미터 업데이트 (SGD 1 step)

optimizer 없이 SGD 1 step을 수동으로 실행하여 forward → loss → backward → update 흐름을 확인한다.

In [ ]:
np.random.seed(42)
model_s = Sequential(
    Linear(4, 8, seed=0),
    Sigmoid(),
    Linear(8, 3, seed=1),
)
lr = 0.01
x_s = np.random.randn(8, 4).astype(np.float32)
t_s = np.eye(3, dtype=np.float32)[np.arange(8) % 3]

losses = []
for step in range(20):
    logits_s = model_s.forward(x_s)
    loss_s = cross_entropy(logits_s, t_s)
    losses.append(float(loss_s))

    grad_s = cross_entropy_grad(logits_s, t_s)
    model_s.backward(grad_s)

    # SGD: param -= lr * grad (in-place)
    for p, g in zip(model_s.params, model_s.grads):
        p -= lr * g

print(f"step  0 loss: {losses[0]:.4f}")
print(f"step 19 loss: {losses[-1]:.4f}  <- 감소해야 함")

plt.figure(figsize=(6, 3))
plt.plot(losses, color='steelblue', linewidth=2)
plt.title("Manual SGD — Cross Entropy Loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Shape 추적 표

784→256→128→10 MLP의 각 레이어를 통과하는 텐서 shape를 단계별로 확인한다.

In [ ]:
model_track = Sequential(
    Linear(784, 256, seed=0),
    Sigmoid(),
    Linear(256, 128, seed=1),
    Sigmoid(),
    Linear(128, 10, seed=2),
)

N = 16
x_track = np.random.randn(N, 784).astype(np.float32)

layer_names = ["Linear(784→256)", "Sigmoid", "Linear(256→128)", "Sigmoid", "Linear(128→10)"]
h = x_track
print(f"{'Layer':<20} {'Input shape':<20} {'Output shape':<20}")
print("-" * 60)
print(f"{'(input)':<20} {'-':<20} {str(h.shape):<20}")
for name, layer in zip(layer_names, model_track.layers):
    in_shape = h.shape
    h = layer.forward(h)
    print(f"{name:<20} {str(in_shape):<20} {str(h.shape):<20}")

## 8. train/eval 모드 전파

`Sequential.train()` / `Sequential.eval()`은 하위 레이어 전체에 모드를 전파한다.
Dropout 등 모드에 따라 동작이 달라지는 레이어에 사용한다.

In [ ]:
model_te = Sequential(
    Linear(4, 8, seed=0),
    Sigmoid(),
    Linear(8, 3, seed=1),
)

model_te.train()
print("train() 후:")
for i, layer in enumerate(model_te.layers):
    print(f"  layer[{i}] ({type(layer).__name__}) training={layer.training}")

model_te.eval()
print("\neval() 후:")
for i, layer in enumerate(model_te.layers):
    print(f"  layer[{i}] ({type(layer).__name__}) training={layer.training}")

## 9. 검증

In [ ]:
# Linear
lin = Linear(6, 4, seed=0)
x_t = np.random.randn(8, 6).astype(np.float32)
out_t = lin.forward(x_t)
assert out_t.shape == (8, 4), f"expected (8,4), got {out_t.shape}"
dout_t = np.ones((8, 4), dtype=np.float32)
dx_t = lin.backward(dout_t)
assert dx_t.shape == (8, 6), f"expected (8,6), got {dx_t.shape}"
assert lin.grad_w.shape == (6, 4)
assert lin.grad_b.shape == (4,)
assert lin.grads[0] is lin.grad_w, "grads[0] must be grad_w reference"

# Sigmoid
sig_t = Sigmoid()
x_t = np.zeros((3, 3), dtype=np.float32)
out_t = sig_t.forward(x_t)
dx_t = sig_t.backward(np.ones_like(x_t))
assert np.allclose(out_t, 0.5), "sigmoid(0) must be 0.5"
assert np.allclose(dx_t, 0.25, atol=1e-5), "sigmoid'(0) must be 0.25"

# ReLU
r_t = ReLU()
x_t = np.array([[-1.0, 0.0, 1.0]], dtype=np.float32)
out_t = r_t.forward(x_t)
dx_t = r_t.backward(np.ones_like(x_t))
assert out_t[0, 0] == 0.0 and out_t[0, 2] == 1.0
assert dx_t[0, 0] == 0.0 and dx_t[0, 2] == 1.0

# Sequential
seq = Sequential(Linear(4, 8, seed=0), Sigmoid(), Linear(8, 3, seed=1))
assert len(seq.params) == 4, f"expected 4 params, got {len(seq.params)}"
x_t = np.random.randn(5, 4).astype(np.float32)
out_t = seq.forward(x_t)
assert out_t.shape == (5, 3)

print("all layer validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 클래스 | forward | backward | params/grads |
|---|---|---|---|
| `Linear` | $xW + b$, shape `(N, out)` | `grad_w`, `grad_b`, `dx` chain rule | `[w, b]` / `[grad_w, grad_b]` |
| `Sigmoid` | $\sigma(x)$, `_out` 저장 | `dout * out * (1-out)` | 없음 |
| `ReLU` | `max(0, x)`, `_mask` 저장 | `dout * _mask` | 없음 |
| `Sequential` | 순방향 레이어 체인 | 역방향 레이어 체인 | 하위 레이어 자동 집계 |

**핵심 설계 원칙**
- `Linear.backward`는 `grad_w[...] =`으로 in-place 대입하여 `grads` 리스트의 배열 참조를 유지한다.
- `Sequential`은 생성자에서 `params.extend(layer.params)`로 모든 파라미터를 수집하므로 optimizer는 `model.params` 하나로 모든 파라미터를 업데이트한다.